## Test Ephemeris Fitting with Various Ephemeris Types

In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import os

# my modules
from src.keplarian_ephemeris import *
from src.orbit_manager import *

## Step1: Propagate the Spacecraft

In [ ]:
# settings
# initial tome
orbit = "LCRNS"
# orbit = "Moonlight"
# orbit = "LNSS"
# orbit = 'Polar'
# orbit = 'NRHO'
# orbit = 'DRO

# time settings
if (
    orbit == "ELFO" or orbit == "Polar" or orbit == "CLFO" or orbit == "LCRNS"
):  # or orbit == "Moonlight" or orbit == "LNSS":
    dt = 0.1
    n_period = 3
    sphm = [80, 80]  # Spherical harmonic model degree and order
else:
    dt = 10.0
    n_period = 2
    sphm = [8, 8]  # Spherical harmonic model degree and order

add_earth = True
add_sun = True

dyn = pnt.NBodyDynamics()
dyn.set_integrator(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-12, reltol=1e-12))
dyn.add_body(pnt.Body.Moon(sphm[0], sphm[1]))
if add_earth:
    dyn.add_body(pnt.Body.Earth())
if add_sun:
    dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_time_step(1.0)  # propagation timestep

In [ ]:
basedir = "/Users/keidaiiiyama/Documents/sw_navlab/LuPNT-private/output/Ephemeris/"
orbm_save_dir = basedir + "data/orbits"

orbm = OrbitManager(
    orbit, dyn, n_period=n_period, dt=dt, data_dir=orbm_save_dir, overwrite=False
)

In [ ]:
if not os.path.exists(basedir + "figures/orbit/"):
    os.makedirs(basedir + "figures/orbit/")

if orbit == "NRHO" or orbit == "LNSS" or orbit == "Moonlight":
    inv = 6
else:
    inv = 600

orbm.plot_orbit(
    figname=basedir + "figures/orbit/{}.pdf".format(orbit),
    inv=inv,
    use_black_back=False,
)

In [ ]:
orbm.plot_coe(plot_period=True)

## Step2: Extract the portion of the orbit and fit ephemeris

### Extract Ephemeris

In [ ]:
from src.ephemeris_sim import EphemerisSimulation

esim_dir = basedir + "data/ephemeris/"
esim = EphemerisSimulation(data_dir=esim_dir)

In [ ]:
# fit_mins = [30, 60, 120, 240, 480]
fit_mins = [120]

dt_fit = 60.0
dt_eval = 1.0

esim.setup_orbit(
    orbm,
    sample_M=30,
    fit_mins=fit_mins,
    dt_fit=dt_fit,
    dt_eval=dt_eval,
    use_cheby_sampling=False,
    overwrite=True,
)

### First Test for one scenario

In [ ]:
from src.cartesian_ephemeris import CartesianEphemeris
from src.keplarian_ephemeris import KeplarianEphemeris


def test_fitting(
    ephem_type,
    config,
    esim,
    M,
    fmin,
    plot_init=False,
    use_rtn=True,
    plot_diff=False,
    plot_params=False,
    use_grad_vel=False,
    ylim_pos=None,
    ylim_vel=None,
):
    if ephem_type == "keplarian":
        esim.setup_kepdict(config)
        eph = KeplarianEphemeris(esim.kep_dict, body=pnt.MOON, print_info=True)

    elif ephem_type == "cartesian":
        eph = CartesianEphemeris(
            order=config["order"],
            use_kep=config["use_kep"],
            use_rsw=config["use_rsw"],
            use_fourier=config["use_fourier"],
            use_meq=config["use_meq"],
            poly_type=config["poly_type"],
            convert_to_coe=True,
            body=pnt.MOON,
        )

    else:
        raise ValueError("Invalid ephemeris type. Choose 'cartesian' or 'polynomial'.")

    # fit ephemeris and evaluate error
    M_diff = abs(M - esim.M_array)
    M = esim.M_array[np.argmin(M_diff)]
    print("M:", M)
    t_data = esim.orbdata[M][fmin]["t_data"]
    rvbf = esim.orbdata[M][fmin]["rvbf"]

    t_data_eval = esim.orbdata[M][fmin]["t_data_eval"]
    rvbf_eval = esim.orbdata[M][fmin]["rvbf_eval"]
    rvbf_w_eval = esim.orbdata[M][fmin]["rvbf_w_eval"]

    # fit the ephemeris
    ephem_x = eph.fit(t_data, rvbf, fit_obj="lsq", print_result=True)

    if plot_init:
        ylim_pos = None
        ylim_vel = None
    else:
        if ylim_pos is None:
            ylim_pos = 10
            ylim_vel = 20

    # evaluate error
    eph.plot_fit_error(
        t_data_eval,
        rvbf_eval,
        rvbf_w_eval,
        ephem_x,
        plot_init=plot_init,
        print_stats=True,
        plot_diff=plot_diff,
        use_grad_for_velfit=use_grad_vel,
        ylim_pos=ylim_pos,
        ylim_vel=ylim_vel,
        use_rtn=use_rtn,
        t_data_fit=t_data,
    )

    if plot_params and config["use_kep"]:
        eph.plot_parmas(t_data_eval, rvbf_eval, ephem_x)

In [ ]:
config = {
    "order": 3,
    "use_kep": True,
    "use_rsw": False,
    "use_fourier": False,
    "use_meq": True,
    "poly_type": "chebyshev",
}

test_fitting(
    ephem_type="cartesian",
    config=config,
    esim=esim,
    M=0.0,
    fmin=120,
    plot_init=False,
    use_rtn=True,
    plot_diff=True,
    plot_params=True,
    use_grad_vel=False,
)

In [ ]:
config = {
    "order": 8,
    "use_kep": True,
    "use_rsw": False,
    "use_fourier": False,
    "use_meq": False,
    "poly_type": "chebyshev",
}

test_fitting(
    ephem_type="cartesian",
    config=config,
    esim=esim,
    M=180.0,
    fmin=120,
    plot_init=False,
    use_grad_vel=True,
    use_rtn=True,
    plot_diff=True,
    plot_params=True,
)

### Run all scenarios

In [ ]:
order = 8

config = {
    "order": order,
    "use_kep": True,
    "use_rsw": False,
    "use_fourier": False,
    "use_meq": False,
    "poly_type": "chebyshev",
    "sampling_type": "cheby",
}
ephem_config = esim.fit_ephemeris(
    ephem_type="cartesian",
    config=config,
    print_errors=True,
    print_opt_results=False,
    overwrite=True,
)

In [ ]:
print(ephem_config)
esim.compute_datasize(
    configs=[ephem_config],
    fit_mins=fit_mins,
    precision=1e-5,
    debug=True,
    overwrite=True,
)